# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook loads, explores, and processes the FAIR^2 dataset using the `mlcroissant` library following the Croissant schema specification.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Dataset Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}\n")

## 2. Data Overview
Explore available record sets and their field structures. All entities are referenced using their `@id`.

In [ ]:
# List record sets (by @id)
record_sets = [r["@id"] if isinstance(r, dict) and "@id" in r else r for r in getattr(dataset.metadata, 'recordSet', [])]
if not record_sets:
    print("No record sets listed in metadata—attempting to infer from records.")
    record_sets = dataset.available_record_sets()
    print("Record set @id available for loading:")
    for rs in record_sets:
        print(f"- {rs}")
else:
    print("Record sets defined in the Croissant schema:")
    for rs in record_sets:
        print(f"- {rs}")

Let's inspect the fields of each detected record set:

In [ ]:
for rsid in record_sets:
    print(f"\nRecord Set @id: {rsid}")
    try:
        recset_meta = dataset.get_record_set(rsid)
        print(f"Name: {getattr(recset_meta, 'name', 'N/A')}")
        field_ids = []
        if hasattr(recset_meta, 'field'):
            if type(recset_meta.field) == list:
                for f in recset_meta.field:
                    if isinstance(f, dict) and '@id' in f:
                        field_ids.append(f['@id'])
                    elif isinstance(f, str):
                        field_ids.append(f)
            else:
                field_ids = [recset_meta.field]
        print(f"Fields for record set ({len(field_ids)}):")
        for fid in field_ids:
            print(f"  - {fid}")
    except Exception as e:
        print(f"  [Could not retrieve field metadata: {e}]")

Let's preview a few records from the main record set(s). We'll start with the first detected one (by `@id`).

In [ ]:
# Show a small preview of a few records for the first (main) record set
preview_record_set = record_sets[0]
print(f"\nSample records from record set @id: {preview_record_set}")
for i, rec in enumerate(dataset.records(record_set=preview_record_set)):
    print(rec)
    if i >= 2:
        break

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. All keys (including column and field names) are referenced by their `@id`.

In [ ]:
# Load all records into DataFrames
dataframes = {}
for rsid in record_sets:
    recs = list(dataset.records(record_set=rsid))
    if recs:
        dataframes[rsid] = pd.DataFrame(recs)
        print(f"Loaded DataFrame for record set @id: {rsid} with shape {dataframes[rsid].shape}")
    else:
        print(f"No records found for record set @id: {rsid}")

# Display the columns in the primary record set (by @id)
main_rs = record_sets[0]
print(f"Columns in main record set (@id: {main_rs}):")
print(dataframes[main_rs].columns.tolist())
dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping using field `@id`s. In this example, we will select a numeric field with meaningful values—for this mass spectrometry dataset, let's choose a likely field such as protein molecular weight or abundance.

You can adapt the `numeric_field_id` and `group_field_id` below to other column names as appropriate for this dataset.

In [ ]:
# Set up field `@id`s for EDA
df = dataframes[main_rs]

# Try to automatically detect a numeric field—otherwise, set manually
# (You may need to look at the printed columns above and adjust these @ids as needed)
possible_numeric_fields = [c for c in df.columns if df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[c])]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # Fallback—commonly datasets use 'molecular_weight' or 'normalized_abundance' etc as @id
    numeric_field_id = df.columns[0]  # pick first column if unsure

# Pick a group field if available
possible_group_fields = [c for c in df.columns if df[c].dtype == object and len(df[c].unique()) < df.shape[0] and len(df[c].unique()) > 1]
group_field_id = possible_group_fields[0] if possible_group_fields else None

threshold = df[numeric_field_id].quantile(0.5) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (using @id): {len(filtered_df)} rows")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id if it exists
if group_field_id is not None and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped by {group_field_id} (@id):")
    print(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and group-wise means (if grouping field available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (@id)")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
    sns.barplot(x=grouped.index, y=grouped.values)
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (@id)")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded the FAIR^2 dataset from the Croissant schema.
- All data handling referenced entity `@id`s for traceable, schema-consistent processing.
- We inspected available record sets, extracted record fields, and visualized numeric distributions.
- The dataset is suitable for further protein abundance and classification studies.

**Adapt the field `@id`s for more detailed exploration as needed for your analysis.**